# Pandas for Data Science — Hands-on Retail Analytics
Synthetic data, reproducible analysis, and practical validation. Run all cells from top to bottom. Upload this notebook and the two CSVs to the same Colab session if using Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
print("Pandas version:", pd.__version__)

## 1. Load data and understand its grain
One row represents one order. The files must be available in the notebook working directory.

In [ ]:
orders = pd.read_csv("sample_orders.csv", parse_dates=["order_date"])
customers = pd.read_csv("sample_customers.csv", parse_dates=["signup_date"])
print("Orders:", orders.shape, "Customers:", customers.shape)
display(orders.head(), orders.dtypes.to_frame("dtype"))

## 2. Data quality audit
Measure missing values and duplicate keys before changing anything.

In [ ]:
display(orders.isna().sum().to_frame("missing_count"))
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())
assert orders["order_id"].is_unique
assert customers["customer_id"].is_unique
assert orders["quantity"].gt(0).all()
assert orders["discount_pct"].between(0, 1).all()
assert orders["unit_price"].dropna().ge(0).all()

## 3. Selection and filtering

In [ ]:
high_quantity = orders.loc[orders["quantity"].ge(4), ["order_id", "region", "quantity"]]
display(high_quantity.head())
display(orders.sort_values("order_date").head())

## 4. Explicit cleaning decisions
Missing regions become `Unknown`; missing prices are flagged, **not** silently set to zero.

In [ ]:
clean = orders.copy()
clean["region"] = clean["region"].fillna("Unknown")
clean["price_missing"] = clean["unit_price"].isna()
clean["gross_sales"] = clean["quantity"] * clean["unit_price"]
clean["net_sales"] = clean["gross_sales"] * (1 - clean["discount_pct"])
display(clean[["order_id", "gross_sales", "net_sales", "price_missing"]].head(12))

## 5. Define the business metric
Known completed-order sales excludes pending/returned orders and rows with missing prices. These are not audited financial results.

In [ ]:
eligible = clean.loc[(clean["status"] == "Completed") & (~clean["price_missing"])].copy()
print("Eligible orders:", len(eligible))
print("Known completed-order sales: ${:,.2f}".format(eligible["net_sales"].sum()))

## 6. GroupBy: sales by region and category

In [ ]:
by_region = eligible.groupby("region", as_index=False).agg(orders=("order_id", "size"), net_sales=("net_sales", "sum"), avg_order_value=("net_sales", "mean")).sort_values("net_sales", ascending=False)
by_category = eligible.groupby("category", as_index=False).agg(orders=("order_id", "size"), net_sales=("net_sales", "sum")).sort_values("net_sales", ascending=False)
display(by_region, by_category)

## 7. Join customer attributes safely

In [ ]:
joined = eligible.merge(customers, on="customer_id", how="left", validate="many_to_one", indicator=True)
assert len(joined) == len(eligible)
print(joined["_merge"].value_counts())
display(joined.groupby("segment", dropna=False)["net_sales"].agg(["count", "sum", "mean"]))

## 8. Pivot tables

In [ ]:
pivot = eligible.pivot_table(index="region", columns="category", values="net_sales", aggfunc="sum", fill_value=0)
display(pivot.round(2))

## 9. Time series aggregation

In [ ]:
monthly = eligible.assign(month=eligible["order_date"].dt.to_period("M").astype(str)).groupby("month", as_index=False)["net_sales"].sum()
display(monthly)

## 10. Visualize findings

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_region["region"], by_region["net_sales"])
ax.set(title="Known completed-order sales by region", xlabel="Region", ylabel="Sales ($)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 11. Export a reproducible summary

In [ ]:
by_region.to_csv("region_sales_summary.csv", index=False)
print("Saved region_sales_summary.csv")

## 12. Interpret with caveats
Report which region leads based on the output, how many prices were missing, and why returned/pending orders were excluded. Do not generalize synthetic results to real customers.

In [ ]:
leader = by_region.iloc[0]
print(f"Top region: {leader.region}; known completed-order sales: ${leader.net_sales:,.2f}")
print("Orders missing prices:", int(clean["price_missing"].sum()))